In [8]:
import pandas as pd 
from datetime import datetime
import plotly.express as px

In [3]:
!dir

 Volume in drive C is Windows
 Volume Serial Number is 20F0-32A0

 Directory of c:\Users\irsya\Documents\Jiunx_Files\PROJECT\2021-PDU\01_PDU_python\Notebook

20/11/2023  22:29    <DIR>          .
20/11/2023  22:29    <DIR>          ..
24/10/2023  00:19             1.090 00_convertCondaEnv2PIP.ipynb
23/01/2022  17:57            69.146 01_ExploratoryDataAnalysis.ipynb
28/02/2022  19:13           161.037 03a_tempGetLabelData.ipynb
02/02/2022  01:54            22.890 03_LabelledTripDrillData.ipynb
28/02/2022  19:13            19.900 06_listDataAvailability.ipynb
13/05/2022  02:25           415.373 07_CreateDurationCalcFunction.ipynb
02/04/2023  03:02           878.214 08_getDomeAPI copy.html
06/10/2023  02:12           202.727 08_getDomeAPI copy.ipynb
22/10/2023  23:00           194.333 08_getDomeAPI.ipynb
28/02/2022  19:13           103.562 09_RepairSubActivity.ipynb
26/03/2022  23:11           195.341 10_connectPostgreSQL.ipynb
05/03/2023  15:33           286.556 11_VisualizeTrajectory.i

In [4]:
Filename = '../2023-11-20T14-28_exporty.csv'
AcSumDF = pd.read_csv(Filename, delimiter=';')
AcSumDF['StartDateTime'] = pd.to_datetime(AcSumDF['StartDateTime'], format='%d/%m/%Y %H:%M')
AcSumDF['EndDateTime'] = pd.to_datetime(AcSumDF['EndDateTime'], format='%d/%m/%Y %H:%M')
AcSumDF.head(5)

,wid,Date,StartDateTime,EndDateTime,Duration,Hole_Depth_max,Bit_Depth_avg,DrillingMeterage,RotateDrillingDuration,SlideDrillingDuration,...,DrillingMeteragePerStand,InSlip_Treshold,LABEL_ConnectionActivity,PIC,Section,Remarks,Stand Group_Pred,stand_on_bottom,status,LABEL_All
0,243,09/04/2023,2023-04-09 13:00:00,2023-04-09 15:42:00,162.40,110.0,76.2191,0.0,0.0,0.0,...,0.0,54,NaN,MIH,"26""",NaN,NaN,0,FIRM,TRIP IN--Connection
1,243,09/04/2023,2023-04-09 15:42:00,2023-04-09 15:48:00,5.83,110.0,0.0000,0.0,0.0,0.0,...,0.0,54,NaN,MIH,"26""",NaN,NaN,0,FIRM,TRIP IN--Stationary
2,243,09/04/2023,2023-04-09 15:48:00,2023-04-09 15:59:00,10.08,110.0,2.8000,0.0,0.0,0.0,...,0.0,54,NaN,MIH,"26""",NaN,NaN,0,FIRM,TRIP IN--Stationary
3,243,09/04/2023,2023-04-09 15:59:00,2023-04-09 16:01:00,2.00,110.0,3.2500,0.0,0.0,0.0,...,0.0,54,NaN,MIH,"26""",NaN,NaN,0,FIRM,TRIP IN--Stationary
4,243,09/04/2023,2023-04-09 16:01:00,2023-04-09 16:03:00,1.67,110.0,11.0500,0.0,0.0,0.0,...,0.0,54,NaN,MIH,"26""",NaN,NaN,0,FIRM,TRIP IN--Moving


In [26]:
AcSumDF.columns

Index(['wid', 'Date', 'StartDateTime', 'EndDateTime', 'Duration',
       'Hole_Depth_max', 'Bit_Depth_avg', 'DrillingMeterage',
       'RotateDrillingDuration', 'SlideDrillingDuration', 'ReamingDuration',
       'ConnectionDuration', 'OnBottomDurationPerStand', 'StandDuration',
       'LABEL_SubActivity', 'LABEL_Activity', 'DrillingMeteragePerStand',
       'InSlip_Treshold', 'LABEL_ConnectionActivity', 'PIC', 'Section',
       'Remarks', 'Stand Group_Pred', 'stand_on_bottom', 'status',
       'LABEL_All'],
      dtype='object')

In [5]:
import requests
from datetime import datetime
import plotly.express as px
cid = 243
result = dict(requests.get(f"http://khansadev.xyz/dome_api/rtdc/get_realtime_interval/{cid}").json()['result'])
result

{'Start': '2023-04-06 16:52:20', 'End': '2023-06-15 10:43:25'}

In [96]:

Timeline_DF = AcSumDF[['StartDateTime', 'EndDateTime','LABEL_Activity','LABEL_SubActivity']]
Timeline_DF['Label'] = AcSumDF['Section']
Timeline_DF
date_format = '%Y-%m-%d %H:%M:%S'
Realtime_DF = pd.DataFrame.from_dict([{
    'StartDateTime':datetime.strptime(result['Start'], date_format),
    'EndDateTime':datetime.strptime(result['End'], date_format),
    'Label':'Realtime',
}]
)


# px.timeline(pd.concat([Realtime_DF, Timeline_DF]), x_start="StartDateTime", x_end="EndDateTime", y="Label")
fig = px.timeline(Timeline_DF, 
            x_start="StartDateTime", 
            x_end="EndDateTime", 
            y="Label",
            hover_data=['LABEL_Activity','LABEL_SubActivity'],
            color='LABEL_SubActivity'
            )
for i in fig.data:
    # if isinstance(i,go._bar.Bar):
    i.marker.line.width = 0
fig

C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\4272311875.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [97]:
def checkActSum(df):

    df['StartDateTime'] = df['StartDateTime'].astype('datetime64[ns]')
    df['EndDateTime'] = df['EndDateTime'].astype('datetime64[ns]')
    # df['Duration'] = df['Duration'].astype('float')
    # df['Duration'] = pd.to_timedelta(df['Duration'], unit='minutes')
    df = df.sort_values('StartDateTime')
    df = df.reset_index(drop=True)

    df['Diff'] = df['StartDateTime'].shift(-1) - df['EndDateTime']
    # case if Diff > 0
    # df_out = pd.DataFrame(df)
    idx_start = 0
    df_concat_list = []
    df['LABEL_IsVoid'] = False
    i = 0
    for idx,row in df[(df['Diff'] > pd.Timedelta(0))].iterrows():
        print(i)
        if i == 0:
            df_concat_list.append(df.loc[idx_start:idx])
            i = i+1
        else:
            df_concat_list.append(df.loc[idx_start+1:idx])
        new_row = {
            "StartDateTime":row["EndDateTime"],
            "EndDateTime":df.loc[idx+1, 'StartDateTime'],
            "LABEL_SubActivity":"Look and Define",
            "LABEL_IsVoid":True,
        }
        # st.write(new_row)
        df_concat_list.append(pd.DataFrame([new_row]))
        idx_start = idx
    df_concat_list.append(df.loc[idx+1:])

        

    df_out = pd.concat(df_concat_list, ignore_index=True,axis=0)
    return df_out
df_out = checkActSum(Timeline_DF)
display(df_out)


0
1
1


C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\2319637053.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\2319637053.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,StartDateTime,EndDateTime,LABEL_Activity,LABEL_SubActivity,Label,Diff,LABEL_IsVoid
0,2023-04-09 13:00:00,2023-04-09 15:42:00,TRIP IN,Connection,"26""",0 days 00:00:00,False
1,2023-04-09 15:42:00,2023-04-09 15:48:00,TRIP IN,Stationary,"26""",0 days 00:00:00,False
2,2023-04-09 15:48:00,2023-04-09 15:59:00,TRIP IN,Stationary,"26""",0 days 00:00:00,False
3,2023-04-09 15:59:00,2023-04-09 16:01:00,TRIP IN,Stationary,"26""",0 days 00:00:00,False
4,2023-04-09 16:01:00,2023-04-09 16:03:00,TRIP IN,Moving,"26""",0 days 00:00:00,False
5,2023-04-09 16:03:00,2023-04-09 16:30:00,TRIP IN,Connection,"26""",0 days 00:00:00,False
6,2023-04-09 16:30:00,2023-04-09 16:31:00,TRIP IN,Stationary,"26""",0 days 00:00:00,False
7,2023-04-09 16:31:00,2023-04-09 16:32:00,TRIP IN,Moving,"26""",0 days 04:11:00,False
8,2023-04-09 16:32:00,2023-04-09 20:43:00,NaN,Look and Define,NaN,NaT,True
9,2023-04-09 20:43:00,2023-04-09 20:46:00,TRIP IN,Connection,"26""",0 days 00:00:00,False


In [45]:
# display(df_out[df_out['LABEL_IsVoid']])

df_out_2 = df_out
df_out_2['LABEL_IsVoid'] = ~df_out_2['LABEL_IsVoid']

df_fin = pd.concat([df_out.head(1), df_out_2, df_out.tail(1)])
df_fin.loc[(~df_fin['LABEL_IsVoid']).tolist(), 'LABEL_SubActivity'] = 'VOID'
df_fin.loc[df_fin['LABEL_IsVoid'].tolist(), 'LABEL_SubActivity'] = 'DONE'
df_fin['Label'] = 'REALTIME'
display(df_fin)
Realtime_DF = pd.DataFrame.from_dict([{
    'StartDateTime':datetime.strptime(result['Start'], date_format),
    'EndDateTime':datetime.strptime(result['End'], date_format),
    'Label':'Realtime',
}]
)
Realtime_DF['StartDateTime']
if df_fin['StartDateTime'].min() > Realtime_DF['StartDateTime'][0]:
    print(f"{df_fin['StartDateTime'].min()} - {Realtime_DF['StartDateTime'][0]}")






0
1
1


C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\2319637053.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\2319637053.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,StartDateTime,EndDateTime,LABEL_Activity,LABEL_SubActivity,Label,Diff,LABEL_IsVoid
0,2023-04-09 13:00:00,2023-04-09 15:42:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
0,2023-04-09 13:00:00,2023-04-09 15:42:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
1,2023-04-09 15:42:00,2023-04-09 15:48:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
2,2023-04-09 15:48:00,2023-04-09 15:59:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
3,2023-04-09 15:59:00,2023-04-09 16:01:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
4,2023-04-09 16:01:00,2023-04-09 16:03:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
5,2023-04-09 16:03:00,2023-04-09 16:30:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
6,2023-04-09 16:30:00,2023-04-09 16:31:00,TRIP IN,VOID,REALTIME,0 days 00:00:00,False
7,2023-04-09 16:31:00,2023-04-09 16:32:00,TRIP IN,VOID,REALTIME,0 days 04:11:00,False
8,2023-04-09 16:32:00,2023-04-09 20:43:00,NaN,DONE,REALTIME,NaT,True


2023-04-09 13:00:00 - 2023-04-06 16:52:20


In [98]:
import plotly.express as px
fig = px.timeline(pd.concat([ Realtime_DF,Timeline_DF,]), 
            x_start="StartDateTime", 
            x_end="EndDateTime", 
            y="Label",
            hover_data=['LABEL_SubActivity'],
            color='LABEL_SubActivity'
            )
for i in fig.data:
    i.marker.line.width = 0
fig

### Final

In [105]:
import pandas as pd 
import requests

Filename = '../2023-11-20T14-28_exporty.csv'
AcSumDF = pd.read_csv(Filename, delimiter=';')
AcSumDF['StartDateTime'] = pd.to_datetime(AcSumDF['StartDateTime'], format='%d/%m/%Y %H:%M')
AcSumDF['EndDateTime'] = pd.to_datetime(AcSumDF['EndDateTime'], format='%d/%m/%Y %H:%M')


cid = 243
result = dict(requests.get(f"http://khansadev.xyz/dome_api/rtdc/get_realtime_interval/{cid}").json()['result'])
result

{'Start': '2023-04-06 16:52:20', 'End': '2023-06-15 10:43:25'}

In [106]:
def checkActSum(df, Label="Label"):
    df = df.copy()

    df['StartDateTime'] = df['StartDateTime'].astype('datetime64[ns]')
    df['EndDateTime'] = df['EndDateTime'].astype('datetime64[ns]')
    # df['Duration'] = df['Duration'].astype('float')
    # df['Duration'] = pd.to_timedelta(df['Duration'], unit='minutes')
    df = df.sort_values('StartDateTime')
    df = df.reset_index(drop=True)

    df['Diff'] = df['StartDateTime'].shift(-1) - df['EndDateTime']
    # print(df)
    # case if Diff > 0
    # df_out = pd.DataFrame(df)
    idx_start = 0
    df_concat_list = []
    df['LABEL_IsVoid'] = False
    i = 0
    for idx,row in df[(df['Diff'] > pd.Timedelta(0))].iterrows():
        print(i)
        if i == 0:
            df_concat_list.append(df.loc[idx_start:idx])
            i = i+1
        else:
            df_concat_list.append(df.loc[idx_start+1:idx])
        new_row = {
            "StartDateTime":row["EndDateTime"],
            "EndDateTime":df.loc[idx+1, 'StartDateTime'],
            Label:"Look and Define",
            "LABEL_IsVoid":True,
        }
        # st.write(new_row)
        df_concat_list.append(pd.DataFrame([new_row]))
        idx_start = idx
    df_concat_list.append(df.loc[idx+1:])

        

    df_out = pd.concat(df_concat_list, ignore_index=True,axis=0)
    return df_out

'DONE'

In [121]:
Timeline_DF = AcSumDF[['StartDateTime', 'EndDateTime']]
Timeline_DF['Y-Axis'] = AcSumDF['Section']
Timeline_DF['Label'] = AcSumDF['LABEL_SubActivity']


Realtime_DF = checkActSum(Timeline_DF)
Realtime_DF

date_format = '%Y-%m-%d %H:%M:%S'
AllRealtime_DF = pd.DataFrame.from_dict([{
    'StartDateTime':datetime.strptime(result['Start'], date_format),
    'EndDateTime':datetime.strptime(result['End'], date_format),
    'Y-Axis':'Realtime',
}]
)
Realtime_DF['LABEL_IsVoid'] = ~Realtime_DF['LABEL_IsVoid']

# df_fin = pd.concat([Timeline_DF.head(1), Timeline_DF, Timeline_DF.tail(1)])
Realtime_DF.loc[(~Realtime_DF['LABEL_IsVoid']).tolist(), 'Label'] = 'VOID'
Realtime_DF.loc[Realtime_DF['LABEL_IsVoid'].tolist(), 'Label'] = 'DONE'
Realtime_DF['Y-Axis'] = 'REALTIME'
Realtime_DF.drop(['Diff','LABEL_IsVoid'], axis=1, inplace=True)

if AllRealtime_DF['StartDateTime'][0] < Realtime_DF['StartDateTime'].min():
    PreRealtime_DF = pd.DataFrame.from_dict([{
    'StartDateTime':datetime.strptime(result['Start'], date_format),
    'EndDateTime':Realtime_DF.head(1)['StartDateTime'][0],
    'Y-Axis':'REALTIME',
    'Label':'VOID',
        }]
        )
    Realtime_DF = pd.concat([PreRealtime_DF, Realtime_DF], ignore_index=True,axis=0)
if AllRealtime_DF['EndDateTime'][0] > Realtime_DF['EndDateTime'].max():
    PostRealtime_DF = pd.DataFrame.from_dict([{
    'StartDateTime':Realtime_DF.head(1)['EndDateTime'][0],
    'EndDateTime':datetime.strptime(result['End'], date_format),
    'Y-Axis':'REALTIME',
    'Label':'VOID',
        }]
        )
    Realtime_DF = pd.concat([ Realtime_DF, PostRealtime_DF], ignore_index=True,axis=0)
    # StartDateTime	EndDateTime	Y-Axis	Label

0
1
1


C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\1198138119.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\irsya\AppData\Local\Temp\ipykernel_29360\1198138119.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [122]:
import plotly.express as px
fig = px.timeline(pd.concat([ Realtime_DF,Timeline_DF,]), 
            x_start="StartDateTime", 
            x_end="EndDateTime", 
            y="Y-Axis",
            hover_data=['Label'],
            color='Label'
            )
for i in fig.data:
    i.marker.line.width = 0
fig

In [14]:
import requests
import plotly.express as px
cid = 243

def getTimelinePlot(ActSumDF, cid,  TimelineRange='All'):
    def _checkActSum(df, Label="Label"):
        df = df.copy()

        df['StartDateTime'] = df['StartDateTime'].astype('datetime64[ns]')
        df['EndDateTime'] = df['EndDateTime'].astype('datetime64[ns]')
        # df['Duration'] = df['Duration'].astype('float')
        # df['Duration'] = pd.to_timedelta(df['Duration'], unit='minutes')
        df = df.sort_values('StartDateTime')
        df = df.reset_index(drop=True)

        df['Diff'] = df['StartDateTime'].shift(-1) - df['EndDateTime']
        # print(df)
        # case if Diff > 0
        # df_out = pd.DataFrame(df)
        idx_start = 0
        df_concat_list = []
        df['LABEL_IsVoid'] = False
        i = 0
        for idx,row in df[(df['Diff'] > pd.Timedelta(0))].iterrows():
            # print(i)
            if i == 0:
                df_concat_list.append(df.loc[idx_start:idx])
                i = i+1
            else:
                df_concat_list.append(df.loc[idx_start+1:idx])
            new_row = {
                "StartDateTime":row["EndDateTime"],
                "EndDateTime":df.loc[idx+1, 'StartDateTime'],
                Label:"Look and Define",
                "LABEL_IsVoid":True,
            }
            # st.write(new_row)
            df_concat_list.append(pd.DataFrame([new_row]))
            idx_start = idx
        df_concat_list.append(df.loc[idx+1:])

            

        df_out = pd.concat(df_concat_list, ignore_index=True,axis=0)
        return df_out
    
    result = dict(requests.get(f"http://khansadev.xyz/dome_api/rtdc/get_realtime_interval/{cid}").json()['result'])

    date_format = '%Y-%m-%d %H:%M:%S'
    AllRealtime_DF = pd.DataFrame.from_dict([{
        'StartDateTime':datetime.strptime(result['Start'], date_format),
        'EndDateTime':datetime.strptime(result['End'], date_format),
        'Y-Axis':'Realtime',
    }]
    )
    
    Timeline_DF = ActSumDF[['StartDateTime', 'EndDateTime']]
    Timeline_DF['Y-Axis'] = ActSumDF['Section']
    Timeline_DF['Label'] = ActSumDF['LABEL_SubActivity']


    Realtime_DF = _checkActSum(Timeline_DF)


    Realtime_DF['LABEL_IsVoid'] = ~Realtime_DF['LABEL_IsVoid']

    # df_fin = pd.concat([Timeline_DF.head(1), Timeline_DF, Timeline_DF.tail(1)])
    Realtime_DF.loc[(~Realtime_DF['LABEL_IsVoid']).tolist(), 'Label'] = 'VOID'
    Realtime_DF.loc[Realtime_DF['LABEL_IsVoid'].tolist(), 'Label'] = 'DONE'
    Realtime_DF['Y-Axis'] = 'REALTIME'
    Realtime_DF.drop(['Diff','LABEL_IsVoid'], axis=1, inplace=True)
    if  TimelineRange=='All':
        if AllRealtime_DF['StartDateTime'][0] < Realtime_DF['StartDateTime'].min():
            PreRealtime_DF = pd.DataFrame.from_dict([{
            'StartDateTime':datetime.strptime(result['Start'], date_format),
            'EndDateTime':Realtime_DF.head(1)['StartDateTime'][0],
            'Y-Axis':'REALTIME',
            'Label':'VOID',
                }]
                )
            Realtime_DF = pd.concat([PreRealtime_DF, Realtime_DF], ignore_index=True,axis=0)
        if AllRealtime_DF['EndDateTime'][0] > Realtime_DF['EndDateTime'].max():
            PostRealtime_DF = pd.DataFrame.from_dict([{
            'StartDateTime':Realtime_DF.head(1)['EndDateTime'][0],
            'EndDateTime':datetime.strptime(result['End'], date_format),
            'Y-Axis':'REALTIME',
            'Label':'VOID',
                }]
                )
            Realtime_DF = pd.concat([ Realtime_DF, PostRealtime_DF], ignore_index=True,axis=0)
        

    
    fig = px.timeline(pd.concat([ Realtime_DF,Timeline_DF,]), 
                x_start="StartDateTime", 
                x_end="EndDateTime", 
                y="Y-Axis",
                hover_data=['Label'],
                color='Label'
                )
    for i in fig.data:
        i.marker.line.width = 0
    return fig

In [17]:
getTimelinePlot(AcSumDF, cid, TimelineRange='All')


C:\Users\irsya\AppData\Local\Temp\ipykernel_17868\3530276992.py:58: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\irsya\AppData\Local\Temp\ipykernel_17868\3530276992.py:59: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
date_format = '%Y-%m-%d %H:%M:%S'
tes = pd.DataFrame.from_dict([{
    'StartDateTime':datetime.strptime(result['Start'], date_format),
    'EndDateTime':datetime.strptime(result['End'], date_format),
    'Label':'Realtime',
}]
)
display(tes)


tes['EndDateTime'][0]



,StartDateTime,EndDateTime,Label
0,2023-04-06 16:52:20,2023-06-15 10:43:25,Realtime


Timestamp('2023-06-15 10:43:25')